In [1]:
import csv
import time
import random
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

# Name: Muhammad Izzul Danish bin Abdul Rasib
# ID: SW01083596

PRODUCT_URL = "https://tudungpeople.com/products/hijab-ring?variant=44400372154517"


# ----------------------------
# Helpers (polite scraping)
# ----------------------------
def polite_sleep(a: float = 1.0, b: float = 2.5):
    """
    Pause the script for a random amount of time between 'a' and 'b' seconds.

    Purpose:
    - Avoid sending requests too quickly (polite scraping)
    - Reduce the chance of being rate-limited or blocked by the website
    """
    time.sleep(random.uniform(a, b))


def robots_allows(url: str, user_agent: str = "*") -> bool:
    """
    Check the website's robots.txt file to see if the given URL is allowed to be scraped.

    Purpose:
    - Respect the website rules for automated access
    - Return True if scraping is allowed, False if disallowed
    """
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        # If robots.txt cannot be accessed, return True (user decides the risk)
        return True


def hard_block_sensitive_paths(url: str) -> None:
    """
    Block access to sensitive ecommerce pages (checkout/cart/account/admin).

    Purpose:
    - Prevent accidental scraping of restricted/private areas
    - Protect the script from violating terms of service for sensitive paths
    """
    blocked = ["/checkout", "/checkouts", "/cart", "/carts", "/account", "/admin", "/orders"]
    path = urlparse(url).path.lower()
    if any(path.startswith(b) for b in blocked):
        raise ValueError(f"Blocked URL path for safety/ToS reasons: {path}")


def fetch_html(url: str, session: requests.Session) -> str:
    """
    Send an HTTP GET request to download the HTML content of a webpage.

    Purpose:
    - Retrieve the page source (HTML) so it can be parsed using BeautifulSoup
    - Use headers to simulate a normal browser request
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; PublicReviewsScraper/1.0)",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }
    r = session.get(url, headers=headers, timeout=25)
    r.raise_for_status()  # raise error if status code is not 200
    return r.text


# ----------------------------
# Review parsing
# ----------------------------
@dataclass
class Review:
    """
    A structured data container for a single review.

    Purpose:
    - Store extracted review details in a clean format
    - Later converted to dictionary using asdict() for easier CSV export
    """
    review_id: Optional[str]
    reviewer_name: Optional[str]
    location: Optional[str]
    rating: Optional[int]
    created_at: Optional[str]
    title: Optional[str]
    body: Optional[str]
    product_title: Optional[str]
    product_url: Optional[str]
    verified_buyer: Optional[bool]
    pictures: List[str]


def parse_reviews_from_html(html: str) -> Dict[str, Any]:
    """
    Parse the HTML and extract reviews from the Judge.me review widget.

    Purpose:
    - Identify review sections using CSS selectors
    - Extract reviewer name, date, and review content (and other fields if available)
    - Return results as a dictionary containing:
      - summary data (ratings count, average rating)
      - list of reviews (converted into dictionaries)
    """
    soup = BeautifulSoup(html, "lxml")

    # Extract summary details from the review widget (if present)
    widg = soup.select_one(".jdgm-rev-widg")
    summary = {}
    if widg:
        summary = {
            "average_rating": float(widg.get("data-average-rating")) if widg.get("data-average-rating") else None,
            "number_of_reviews": int(widg.get("data-number-of-reviews")) if widg.get("data-number-of-reviews") else None,
            "updated_at": widg.get("data-updated-at"),
            "image_url": widg.get("data-image-url"),
        }

    reviews: List[Review] = []
    for r in soup.select(".jdgm-rev"):
        # Extract rating value from the review (optional)
        rating = None
        rating_el = r.select_one(".jdgm-rev__rating")
        if rating_el and rating_el.get("data-score"):
            try:
                rating = int(rating_el["data-score"])
            except ValueError:
                rating = None

        # Extract date from timestamp attribute (if available)
        ts_el = r.select_one(".jdgm-rev__timestamp")
        created_at = ts_el.get("data-content") if ts_el else None

        # Extract photo links (optional)
        pics = [a["href"] for a in r.select(".jdgm-rev__pic-link[href]")]

        # Extract verified buyer flag (optional)
        verified_attr = r.get("data-verified-buyer")
        verified = True if verified_attr == "true" else (False if verified_attr == "false" else None)

        # Build review object
        reviews.append(
            Review(
                review_id=r.get("data-review-id"),
                reviewer_name=(r.select_one(".jdgm-rev__author").get_text(strip=True)
                               if r.select_one(".jdgm-rev__author") else None),
                location=(r.select_one(".jdgm-rev__location").get_text(" ", strip=True)
                          if r.select_one(".jdgm-rev__location") else None),
                rating=rating,
                created_at=created_at,
                title=(r.select_one(".jdgm-rev__title").get_text(" ", strip=True)
                       if r.select_one(".jdgm-rev__title") else None),
                body=(r.select_one(".jdgm-rev__body").get_text(" ", strip=True)
                      if r.select_one(".jdgm-rev__body") else None),
                product_title=r.get("data-product-title"),
                product_url=r.get("data-product-url"),
                verified_buyer=verified,
                pictures=pics,
            )
        )

    return {
        "summary": summary,
        "reviews": [asdict(x) for x in reviews],
        "review_count_parsed": len(reviews),
        "has_widget": bool(widg),
    }


def save_reviews_csv(reviews: List[Dict[str, Any]], path: str):
    """
    Save extracted reviews to a CSV file with ONLY 3 required columns.

    Purpose:
    - Match assignment requirement exactly:
      a) Reviewer name
      b) Review date
      c) Review content
    - Write output into a clean CSV format
    """
    fieldnames = ["reviewer_name", "review_date", "review_content"]

    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

        for r in reviews:
            w.writerow({
                "reviewer_name": (r.get("reviewer_name") or "").strip(),
                "review_date": (r.get("created_at") or "").strip(),
                "review_content": (r.get("body") or "").strip(),
            })


def main():
    """
    Main function to run the whole scraping process.

    Purpose:
    - Perform safety checks (block sensitive paths + robots.txt check)
    - Fetch product page HTML
    - Parse the HTML to extract reviews
    - Save the required fields into CSV
    """
    hard_block_sensitive_paths(PRODUCT_URL)

    # Stop scraping if robots.txt likely disallows it
    if not robots_allows(PRODUCT_URL):
        raise RuntimeError("robots.txt likely disallows this URL for your user-agent. Don't scrape it.")

    session = requests.Session()

    # Sleep before request to be polite
    polite_sleep()

    # Fetch HTML, parse reviews, and save to CSV
    html = fetch_html(PRODUCT_URL, session)
    data = parse_reviews_from_html(html)

    print("Has Judge.me widget:", data["has_widget"])
    print("Parsed reviews on page:", data["review_count_parsed"])
    print("Summary:", data["summary"])

    save_reviews_csv(data["reviews"], "tudungpeople_reviews.csv")
    print("Saved: tudungpeople_reviews.csv")


if __name__ == "__main__":
    main()

Has Judge.me widget: True
Parsed reviews on page: 8
Summary: {'average_rating': 4.94, 'number_of_reviews': 33, 'updated_at': '2026-02-19T12:23:06Z', 'image_url': 'https://cdn.shopify.com/s/files/1/0260/2821/2269/files/TudungPeople-Hijab-Ring-Campaign-00.jpg?v=1710991151'}
Saved: tudungpeople_reviews.csv
